In [1]:
# src/cgx/ast/parse_codebase.py
from __future__ import annotations

import ast
import io
import os
import tokenize
from typing import Any, Dict, List, Optional, Tuple

In [2]:
project_root = "/home/rmohammadi/Downloads/ibase-t-poc1"

In [3]:
def compute_module_path(project_root: str, file_path: str) -> str:
    """
    Compute a deterministic dotted module path for a Python file within a project.

    Rules (purely deterministic; no runtime inspection of __init__.py):
      1) Normalize paths and compute relpath from project_root.
      2) Strip the trailing ".py".
      3) Replace path separators with dots.
      4) If the file is "__init__.py", drop the final segment (package module path).

    Examples
    --------
    project_root=/repo
      /repo/pkg/mod.py           -> "pkg.mod"
      /repo/pkg/__init__.py      -> "pkg"
      /repo/app/sub/utils.py     -> "app.sub.utils"

    Notes
    -----
    - If file_path is outside project_root, we return a dotted path built from the
      normalized absolute path sans drive letter (on Windows) and log a warning.
    - This function is deterministic and side-effect-free.

    Parameters
    ----------
    project_root : str
        Absolute or relative path to the repository root.
    file_path : str
        Absolute or relative path to a Python source file.

    Returns
    -------
    str
        Dotted module path (may be empty string if relpath cannot be derived).
    """
    try:
        project_root = os.path.abspath(project_root)
        file_path = os.path.abspath(file_path)

        # Defensive normalization for separators
        rel_path = os.path.relpath(file_path, project_root)
        rel_path = rel_path.replace("\\", "/")  # normalize Windows slashes

        # If file is outside root, fallback to absolute sans drive
        if rel_path.startswith(".."):
            logger.warning("File %s is outside project root %s", file_path, project_root)
            drive, tail = os.path.splitdrive(file_path)
            rel_path = tail.lstrip(os.sep).replace("\\", "/")

        # Strip extension
        if rel_path.endswith(".py"):
            rel_path = rel_path[:-3]

        # Replace separators with dots
        mod_path = rel_path.replace("/", ".")

        # Drop trailing .__init__
        if mod_path.endswith(".__init__"):
            mod_path = mod_path.rsplit(".", 1)[0]

        # Guarantee non-empty
        if not mod_path:
            return os.path.splitext(os.path.basename(file_path))[0]

        return mod_path

    except Exception as e:
        return os.path.splitext(os.path.basename(file_path))[0]


In [4]:
for root, _, files in os.walk(project_root):
    for fname in files:
        if not fname.endswith(".py"):
            continue
        else:
            break

In [5]:
fname

'main.py'

In [29]:
filepath = "/home/rmohammadi/Downloads/ibase-t-poc1/business/domain/scanai/pdf_to_md_v2.py"
try:
    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        source_code = f.read()
    try:
        tree = ast.parse(source_code, filename=filepath)
    except SyntaxError:
        pass
except Exception as e:
    pass

module_path = compute_module_path(project_root, filepath)

In [30]:
print(module_path)

business.domain.scanai.pdf_to_md_v2


In [96]:
import ast

def extract_ast_info(node):
    """
    Recursively extract structured information from an AST node.

    Args:
        node (ast.AST): Any Python AST node

    Returns:
        dict | list | str | None: A structured JSON-like description
    """
    if isinstance(node, ast.AST):
        result = {"_type": type(node).__name__}
        for field_name, field_value in ast.iter_fields(node):
            result[field_name] = extract_ast_info(field_value)
        return result
    elif isinstance(node, list):
        return [extract_ast_info(item) for item in node]
    else:
        # Primitive: int, str, None, etc.
        return node



structured = extract_ast_info(tree.body[-3])
print(structured)


{'_type': 'ClassDef', 'name': 'GenerateMarkdown', 'bases': [], 'keywords': [], 'body': [{'_type': 'Expr', 'value': {'_type': 'Constant', 'value': '\n    Generates Markdown output from structured Docling elements using a row-based layout.\n\n    Attributes:\n        structured_pages (list): Page-level structured data.\n        sorted_items (list): Flat list of (type, item, center_y, left) tuples.\n        output_path (str): Path to write the generated Markdown file.\n        row_tolerance (int): Vertical grouping threshold.\n        page_width (int): Width of the page for layout-aware rendering.\n    ', 'kind': None}}, {'_type': 'AnnAssign', 'target': {'_type': 'Name', 'id': 'structured_pages', 'ctx': {'_type': 'Store'}}, 'annotation': {'_type': 'Name', 'id': 'list', 'ctx': {'_type': 'Load'}}, 'value': None, 'simple': 1}, {'_type': 'AnnAssign', 'target': {'_type': 'Name', 'id': 'sorted_items', 'ctx': {'_type': 'Store'}}, 'annotation': {'_type': 'Name', 'id': 'list', 'ctx': {'_type': 'Lo

In [97]:
print(tree.body[-3].body[0].value.value)


    Generates Markdown output from structured Docling elements using a row-based layout.

    Attributes:
        structured_pages (list): Page-level structured data.
        sorted_items (list): Flat list of (type, item, center_y, left) tuples.
        output_path (str): Path to write the generated Markdown file.
        row_tolerance (int): Vertical grouping threshold.
        page_width (int): Width of the page for layout-aware rendering.
    


In [98]:
tree.body

In [ ]:
def _unparse(node: Optional[ast.AST]) -> Optional[str]:
    if node is None:
        return None
    try:
        return ast.unparse(node)
    except Exception:
        return getattr(node, "id", None) or getattr(node, "attr", None) or type(node).__name__

def _class_signature(node: ast.ClassDef) -> str:
    try:
        bases = [_unparse(b) for b in getattr(node, "bases", [])] or []
        base_seg = f"({', '.join(bases)})" if bases else ""
        return f"class {node.name}{base_seg}"
    except Exception:
        return f"class {getattr(node, 'name', 'Unknown')}"

In [70]:
res = _class_signature(tree.body[-3])

In [85]:
import ast
from typing import List

def _signature_str(args: ast.arguments) -> str:
    """
    Construct a string representation of a function's signature from an AST `arguments` node.

    Args:
        args (ast.arguments): The AST node representing the arguments of a function
            definition. This includes positional-only arguments, regular arguments,
            defaults, varargs (*args), keyword-only args, kwarg (**kwargs), and
            optional type annotations.

    Returns:
        str: A string formatted like a Python function parameter list,
        enclosed in parentheses. Returns `"()"` if extraction fails.

    Notes:
        - Positional-only arguments (if present) are followed by `/`.
        - Regular arguments are included, with defaults if present.
        - `*args` and `**kwargs` are included when present.
        - Keyword-only arguments are handled, with or without defaults.
        - Type annotations are included when available using `_unparse`.
        - Falls back to `"()"` if any error occurs during formatting.

    Example:
        For the function:
            def foo(x: int, y=10, *args, z: str = "hi", **kwargs): pass

        The AST arguments would be converted into:
            "(x: int, y=10, *args, z: str=\"hi\", **kwargs)"
    """
    parts: List[str] = []
    try:
        # Positional-only args
        po = getattr(args, "posonlyargs", [])
        for a in po:
            seg = a.arg + (f": {_unparse(a.annotation)}" if a.annotation else "")
            parts.append(seg)
        if po:
            parts.append("/")

        # Regular args with defaults
        reg = list(args.args)
        ndef = len(args.defaults or [])
        for i, a in enumerate(reg):
            ann = f": {_unparse(a.annotation)}" if a.annotation else ""
            if ndef and i >= len(reg) - ndef:
                j = i - (len(reg) - ndef)
                parts.append(f"{a.arg}{ann}={_unparse(args.defaults[j])}")
            else:
                parts.append(f"{a.arg}{ann}")

        # *args
        if args.vararg:
            a = args.vararg
            parts.append(f"*{a.arg}" + (f": {_unparse(a.annotation)}" if a.annotation else ""))
        elif args.kwonlyargs:
            parts.append("*")

        # kwonlyargs (with defaults if provided)
        for a, d in zip(args.kwonlyargs, args.kw_defaults or [None]*len(args.kwonlyargs)):
            seg = a.arg + (f": {_unparse(a.annotation)}" if a.annotation else "")
            if d is not None:
                seg += f"={_unparse(d)}"
            parts.append(seg)

        # **kwargs
        if args.kwarg:
            a = args.kwarg
            parts.append(f"**{a.arg}" + (f": {_unparse(a.annotation)}" if a.annotation else ""))

        return "(" + ", ".join(parts) + ")"
    except Exception:
        return "()"


In [86]:
def _collect_top_level_members(tree: ast.AST) -> Dict[str, List[Dict[str, Any]]]:
    """
    Collect deterministic summaries of top-level members (no nested scanning).
    Returns:
        {
            "functions": [{"name": str, "signature": str}],
            "classes":   [{"name": str, "signature": str}]
        }
    """
    out = {"functions": [], "classes": []}
    try:
        for n in getattr(tree, "body", []):
            if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef)):
                out["functions"].append({
                    "name": n.name,
                    "signature": _signature_str(n.args)
                })
            elif isinstance(n, ast.ClassDef):
                out["classes"].append({
                    "name": n.name,
                    "signature": _class_signature(n)
                })
    except Exception:
        # Be defensive; partial is fine
        pass
    return out

In [89]:
members = _collect_top_level_members(tree.body[-3])
members

{'functions': [{'name': 'generate', 'signature': '(self)'},
  {'name': 'save', 'signature': '(self)'},
  {'name': 'group_into_rows',
   'signature': '(self, items: list, tolerance: int)'},
  {'name': 'render_structured_table',
   'signature': '(self, table_item: Any, image_path: Any)'},
  {'name': 'compute_intersection_area', 'signature': '(self, b1, b2)'},
  {'name': 'bbox_area', 'signature': '(self, b)'},
  {'name': 'overlap_ratio', 'signature': '(self, text_bbox, other_bbox)'},
  {'name': 'is_text_inside_any_images_',
   'signature': '(self, text_bbox, threshold=0.15)'},
  {'name': '_is_text_inside_any', 'signature': '(self, item)'},
  {'name': 'docling_bbox_to_pymupdf', 'signature': '(self, bbox_bottomleft)'},
  {'name': 'layout_verfied', 'signature': '(self, bbox)'},
  {'name': 'render_cell',
   'signature': '(self, item_type: str, item: Any, left: float, image_path: str, bbox: Any)'},
  {'name': '_render_markdown', 'signature': '(self)'}],
 'classes': []}

In [104]:
def _value_preview(node: Optional[ast.AST], maxlen: int = 160) -> Optional[str]:
    try:
        if node is None:
            return None
        s = _unparse(node) or ""
        if s and len(s) > maxlen:
            return s[: maxlen - 3] + "..."
        return s or None
    except Exception:
        return None

In [105]:
def _get_source(source: str, node: ast.AST) -> str:
    try:
        seg = ast.get_source_segment(source, node)
        return seg if seg is not None else ""
    except Exception:
        return ""


In [106]:
def _collect_top_level_members(tree: ast.AST, source: str) -> Dict[str, List[Dict[str, Any]]]:
    """
    Collect deterministic summaries of top-level members of a file, plus
    imports and globals. This ensures file-level meta is informative even if
    there is no module docstring.

    Returns:
        {
          "functions": [{"name": str, "signature": str, "docstring": str|None}],
          "classes":   [{"name": str, "signature": str, "docstring": str|None}],
          "imports":   [str],
          "globals":   [{"name": str, "value": str|None, "annotation": str|None}]
        }
    """
    out = {"functions": [], "classes": [], "imports": [], "globals": []}
    try:
        for n in getattr(tree, "body", []):
            if isinstance(n, (ast.FunctionDef, ast.AsyncFunctionDef)):
                out["functions"].append({
                    "name": n.name,
                    "signature": _signature_str(n.args),
                    "docstring": ast.get_docstring(n)
                })
            elif isinstance(n, ast.ClassDef):
                out["classes"].append({
                    "name": n.name,
                    "signature": _class_signature(n),
                    "docstring": ast.get_docstring(n)
                })
            elif isinstance(n, (ast.Import, ast.ImportFrom)):
                src = _get_source(source, n)
                if src:
                    out["imports"].append(src.strip())
            elif isinstance(n, ast.Assign) and all(isinstance(t, ast.Name) for t in n.targets):
                out["globals"].append({
                    "name": n.targets[0].id,
                    "value": _value_preview(n.value),
                    "annotation": None
                })
            elif isinstance(n, ast.AnnAssign) and isinstance(n.target, ast.Name):
                out["globals"].append({
                    "name": n.target.id,
                    "value": _value_preview(n.value),
                    "annotation": _unparse(n.annotation)
                })
    except Exception:
        # Be defensive; partial is fine
        pass
    return out


In [107]:
def _build_file_code_stub(module_doc: Optional[str], members: Dict[str, List[Dict[str, Any]]]) -> str:
    """
    Deterministic, compact text summary of a file:
    - Triple-quoted module docstring (if any)
    - Imports (one per line)
    - One-line stubs for globals, functions, classes
    """
    parts: List[str] = []
    if module_doc:
        parts.append('"""' + module_doc.replace('"""', r'\"\"\"') + '"""')

    for imp in members.get("imports", []):
        parts.append(imp)

    for g in members.get("globals", []):
        ann = f": {g['annotation']}" if g.get("annotation") else ""
        val = f" = {g['value']}" if g.get("value") else ""
        parts.append(f"{g['name']}{ann}{val}")

    for f in members.get("functions", []):
        parts.append(f"def {f['name']}{f['signature']}: ...")

    for c in members.get("classes", []):
        sig = c["signature"]  # already 'class Name(Base, ...)'
        parts.append(f"{sig}: ...")

    return "\n".join(parts)


In [109]:
try:
    module_doc = ast.get_docstring(tree)
except Exception:
    module_doc = None
    
members = _collect_top_level_members(tree, source_code)
file_code_stub = _build_file_code_stub(module_doc, members)

In [110]:
members

{'functions': [{'name': '_rect_intersection_area',
   'signature': '(a, b)',
   'docstring': None},
  {'name': '_rect_area', 'signature': '(bb)', 'docstring': None},
  {'name': '_is_corner_image',
   'signature': '(img_bb, page_w, page_h, frac=0.12, cover_thr=0.7)',
   'docstring': 'Returns True if image bbox is mostly inside a page corner.\n  - frac: corner box size as fraction of page (width & height)\n  - cover_thr: required fraction of image area covered by a corner box'},
  {'name': 'extract_docling_layout_and_images',
   'signature': '(input_path, output_dir=cfg.scanai_dir, page_range=None)',
   'docstring': 'Extracts layout from a PDF document using Docling.\n\nArgs:\n    input_path (str or Path): Path to the input PDF file.\n    output_dir (str or Path): Directory where output images will be saved.\n    page_range (tuple[int, int], optional): Page range (start, end) to process. If None, processes all pages.\n\nReturns:\n    tuple[dict, Document]: \n        - structured_pages: A

In [111]:
print(file_code_stub)

import base64
from datetime import datetime
import json
import time, os, re
from pathlib import Path
from PIL import Image
from io import BytesIO
from collections import defaultdict
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions, EasyOcrOptions, TableFormerMode
from docling.document_converter import DocumentConverter, PdfFormatOption
from pathlib import Path
from dataclasses import dataclass, field
from typing import List, Tuple, Set, Union, Any
from foundation import logger
from foundation import config
from math import isclose
from .pdf_analyisis import extract_document_layout
cfg = config.get_config()
log = logger.get_logger()
IMAGE_RESOLUTION_SCALE = 3.0
def _rect_intersection_area(a, b): ...
def _rect_area(bb): ...
def _is_corner_image(img_bb, page_w, page_h, frac=0.12, cover_thr=0.7): ...
def extract_docling_layout_and_images(input_path, output_dir=cfg.scanai_dir, page_range=None): ...
def is_header_footer(e

In [113]:
code_chunks = {
        "id": filepath,
        "type": "file",
        "name": os.path.basename(filepath),
        "file": filepath,
        "module_path": module_path,
        "code": file_code_stub,
        "meta": {
            "docstring": module_doc,
            "members": members,
            "metrics": {
                "n_loc": len(source_code.splitlines())
            }
        }
    }

In [114]:
code_chunks

{'id': '/home/rmohammadi/Downloads/ibase-t-poc1/business/domain/scanai/pdf_to_md_v2.py',
 'type': 'file',
 'name': 'pdf_to_md_v2.py',
 'file': '/home/rmohammadi/Downloads/ibase-t-poc1/business/domain/scanai/pdf_to_md_v2.py',
 'module_path': 'business.domain.scanai.pdf_to_md_v2',
 'code': 'import base64\nfrom datetime import datetime\nimport json\nimport time, os, re\nfrom pathlib import Path\nfrom PIL import Image\nfrom io import BytesIO\nfrom collections import defaultdict\nfrom docling.datamodel.base_models import InputFormat\nfrom docling.datamodel.pipeline_options import PdfPipelineOptions, EasyOcrOptions, TableFormerMode\nfrom docling.document_converter import DocumentConverter, PdfFormatOption\nfrom pathlib import Path\nfrom dataclasses import dataclass, field\nfrom typing import List, Tuple, Set, Union, Any\nfrom foundation import logger\nfrom foundation import config\nfrom math import isclose\nfrom .pdf_analyisis import extract_document_layout\ncfg = config.get_config()\nlog = 

In [115]:
def _comments_by_line(source: str) -> Dict[int, List[str]]:
    result: Dict[int, List[str]] = {}
    try:
        for tok in tokenize.generate_tokens(io.StringIO(source).readline):
            if tok.type == tokenize.COMMENT:
                result.setdefault(tok.start[0], []).append(tok.string)
    except Exception:
        pass
    return result

In [116]:
comments_map = _comments_by_line(source_code)

In [117]:
comments_map

{19: ['# =============================================================================='],
 24: ['# =============================================================================='],
 26: ['# Scale to control resolution of rendered images'],
 29: ['# --- helpers (put above the loop) ---'],
 31: ['# a, b: (l, t, r, b) in BOTTOMLEFT coords'],
 51: ['# top-left'],
 52: ['# top-right'],
 53: ['# bottom-left'],
 54: ['# bottom-right'],
 122: ['# Step 2: Save full-page images'],
 148: ['# Step 3: Save figure images and extract layout'],
 159: ["# --- page size for this picture's page ---"],
 161: ['# 1-based'],
 162: ['# Get page width/height from the page bbox'],
 166: ['# Image bbox (BOTTOMLEFT): (l, t, r, b)'],
 170: ['# Skip logos/images placed in page corners'],
 174: ['# -----------------------------------------'],
 196: ['# Step 4: Save table images and extract layout'],
 259: ['# rows = self.process()'],
 346: ['# BOTTOMLEFT'],
 348: ['# t - b'],
 352: ['# along Y'],
 362: ['# 1) sort

In [121]:
ast.get_source_segment(source_code, tree.body[-3])

'class GenerateMarkdown:\n    """\n    Generates Markdown output from structured Docling elements using a row-based layout.\n\n    Attributes:\n        structured_pages (list): Page-level structured data.\n        sorted_items (list): Flat list of (type, item, center_y, left) tuples.\n        output_path (str): Path to write the generated Markdown file.\n        row_tolerance (int): Vertical grouping threshold.\n        page_width (int): Width of the page for layout-aware rendering.\n    """\n    structured_pages: list\n    sorted_items: list\n    page_layout: list\n    seen_tables = set()\n    results: dict = field(default_factory=dict)\n    output_path: str = "docling_output.md"\n    row_tolerance: int = 15\n    page_width: int = 800\n    page_height: int = 800\n\n    def generate(self) -> str:\n        """\n        Generates Markdown from the sorted items using row alignment and returns the string.\n        """\n        try:\n            self.image_bboxes = [item.prov[0].bbox for t,

In [ ]:
class CodeVisitor(ast.NodeVisitor):
    """
    AST visitor that traverses a Python module and extracts structured
    information about classes, functions, methods, lambdas, and their metadata.

    Attributes
    ----------
    filename : str
        Absolute path of the file being parsed.
    module_path : str
        Dotted import path of the file relative to project root.
    source : str
        Raw source code of the file.
    comments_map : dict[int, list[str]]
        Maps line numbers to associated comment strings.
    current_func_id : str | None
        ID of the function currently being visited.
    current_class_name : str | None
        Name of the class currently being visited (for nested class context).
    func_meta : dict[str, dict]
        Collected metadata for each function/method keyed by ID.
    func_index : dict[str, int]
        Index of function/method chunks inside `code_chunks`.
    import_alias : dict[str, str]
        Mapping from local import alias → fully qualified name.
    star_imports : list[str]
        List of modules imported with `from x import *`.
    """
    def __init__(self, filename: str, module_path: str, source: str):
        """
        Initialize the visitor with file path, module path, and source code.
        """
        super().__init__()
        self.filename = filename
        self.module_path = module_path
        self.source = source
        self.comments_map = _comments_by_line(source)
        self.current_func_id: Optional[str] = None
        self.current_class_name: Optional[str] = None
        self.func_meta: Dict[str, Dict[str, Any]] = {}
        self.func_index: Dict[str, int] = {}
        self.import_alias: Dict[str, str] = {}
        self.star_imports: List[str] = []

    # -------- imports --------
    def visit_Import(self, node: ast.Import):
        """
        Capture `import x [as y]` statements.

        Updates `import_alias` to map alias → original module.
        """
        for alias in node.names:
            asname = alias.asname or alias.name
            self.import_alias[asname] = alias.name
        self.generic_visit(node)

    def visit_ImportFrom(self, node: ast.ImportFrom):
        """
        Capture `from x import y [as z]` and star-imports.

        Updates `import_alias` and `star_imports`.
        """
        mod = node.module or ""
        for alias in node.names:
            if alias.name == "*":
                self.star_imports.append(mod)
            else:
                full = f"{mod}.{alias.name}" if mod else alias.name
                asname = alias.asname or alias.name
                self.import_alias[asname] = full
        self.generic_visit(node)

    # -------- classes --------
    def visit_ClassDef(self, node: ast.ClassDef):
        """
        Visit a class definition and emit a class chunk.

        - Captures class decorators, bases, keywords, docstring.
        - Marks dataclass fields if decorated with @dataclass.
        - Tracks enclosing class context for nested classes.
        - Visits all methods and nested classes recursively.
        """
        class_id = f"{self.filename}::class::{node.name}"
        # Get source code segment of the source that generated node.
        class_code = _get_source(self.source, node) 
        # unparse an AST node decorator_list into its string representations.
        decorators = [_unparse(d) for d in node.decorator_list]

        # Capture enclosing class deterministically (for nested classes)
        enclosing = self.current_class_name

        meta: Dict[str, Any] = {
            "decorators": decorators,
            "bases": [_unparse(b) for b in node.bases],
            "keywords": {(kw.arg or ""): _unparse(kw.value) for kw in getattr(node, "keywords", [])}
            if getattr(node, "keywords", None)
            else {},
            "docstring": ast.get_docstring(node),
            "doc_parsed": _parse_docstring(ast.get_docstring(node)), #Parse a Python docstring into structured sections.
            "is_dataclass": any("dataclass" in (d or "") for d in decorators),
            "dataclass_fields": [],  # filled if dataclass with AnnAssigns
            "enclosing_class": enclosing,  # ★ NEW
        }
        if meta["is_dataclass"]:
            for stmt in node.body:
                if isinstance(stmt, ast.AnnAssign) and isinstance(stmt.target, ast.Name):
                    meta["dataclass_fields"].append(
                        {
                            "name": stmt.target.id,
                            "annotation": _unparse(stmt.annotation),
                            "default": _value_preview(stmt.value),
                        }
                    )

        idx = len(code_chunks)
        code_chunks.append(
            {
                "id": class_id,
                "type": "class",
                "name": node.name,
                "file": self.filename,
                "module_path": self.module_path,  # ★ NEW
                "code": class_code,
                "meta": meta,
            }
        )

        prev_class = self.current_class_name
        self.current_class_name = node.name

        # methods & nested
        for child in node.body:
            if isinstance(child, (ast.FunctionDef, ast.AsyncFunctionDef)):
                self._visit_function_like(child, is_method=True)
            elif isinstance(child, ast.ClassDef):
                # Nested class
                self.visit(child)
            else:
                self.visit(child)

        self.current_class_name = prev_class

    # -------- functions --------
    def visit_FunctionDef(self, node: ast.FunctionDef):
        """
        Visit a synchronous function definition (not a method).
        """
        self._visit_function_like(node, is_method=False)

    def visit_AsyncFunctionDef(self, node: ast.AsyncFunctionDef):
        """
        Visit an asynchronous function definition (not a method).
        """
        self._visit_function_like(node, is_method=False, is_async=True)

    def _detect_method_kind(self, decorators: List[str]) -> str:
        """
        Infer whether a function inside a class is a property,
        staticmethod, classmethod, or instance method.
        """
        if any(d.endswith(".setter") or d.endswith(".deleter") for d in decorators):
            return "property_accessor"
        if any(d.endswith(".getter") or d == "property" for d in decorators):
            return "property"
        if any(d.endswith("staticmethod") or d == "staticmethod" for d in decorators):
            return "staticmethod"
        if any(d.endswith("classmethod") or d == "classmethod" for d in decorators):
            return "classmethod"
        return "instance"

    def _visit_function_like(self, node, is_method: bool, is_async: bool = False):
        """
        Common handler for FunctionDef and AsyncFunctionDef.

        - Builds function/method ID, code, and metadata.
        - Captures signature, decorators, parameters, return annotation,
          docstring, and comments.
        - Initializes per-function metadata in `func_meta`.
        - Visits body recursively and finalizes metrics.
        """
        effective_is_method = is_method or (self.current_class_name is not None)
        qual = (
            f"{self.current_class_name}.{node.name}"
            if effective_is_method and self.current_class_name
            else node.name
        )
        func_id = (
            f"{self.filename}::method::{qual}"
            if effective_is_method
            else f"{self.filename}::function::{node.name}"
        )

        func_code = _get_source(self.source, node)
        decorators = [_unparse(d) for d in node.decorator_list]
        method_kind = self._detect_method_kind(decorators)
        doc = ast.get_docstring(node)
        doc_parsed = _parse_docstring(doc)

        meta: Dict[str, Any] = {
            "decorators": decorators,
            "method_kind": method_kind,
            "is_async": bool(is_async or isinstance(node, ast.AsyncFunctionDef)),
            "is_method": bool(effective_is_method),
            "class_name": self.current_class_name if effective_is_method else None,
            "signature": _signature_str(node.args),
            "parameters": _param_list(node.args),
            "args_struct": {
                "posonlyargs": [_serialize_arg(x) for x in getattr(node.args, "posonlyargs", [])],
                "args": [_serialize_arg(x) for x in node.args.args],
                "vararg": _serialize_arg(node.args.vararg) if node.args.vararg else None,
                "kwonlyargs": [_serialize_arg(x) for x in node.args.kwonlyargs],
                "kw_defaults": [_unparse(x) for x in node.args.kw_defaults] if node.args.kw_defaults else [],
                "kwarg": _serialize_arg(node.args.kwarg) if node.args.kwarg else None,
                "defaults": [_unparse(x) for x in node.args.defaults] if node.args.defaults else [],
            },
            "returns_annotation": _unparse(getattr(node, "returns", None))
            if getattr(node, "returns", None)
            else None,
            "type_comment": getattr(node, "type_comment", None),
            "docstring": doc,
            "doc_parsed": doc_parsed,
            "is_generator": False,
            "return_values": [],
            "yield_values": [],
            "raises": [],
            "exceptions_handled": [],
            "attributes_used": set(),
            "names_used": set(),
            "imports_used": {},
            "lambda_ids": [],
            "calls_detailed": [],
            "local_vars": [],
            "instance_attributes": [],
            "metrics": {
                "n_loc": (getattr(node, "end_lineno", node.lineno) - node.lineno + 1),
                "n_params": len(_param_list(node.args)),
                "n_returns": 0,
                "n_yields": 0,
                "n_branches": 0,
                "n_calls": 0,
            },
            "comments": _comments_in_span(
                self.comments_map,
                getattr(node, "lineno", 0),
                getattr(node, "end_lineno", getattr(node, "lineno", 0)),
            ),
        }

        idx = len(code_chunks)
        code_chunks.append(
            {
                "id": func_id,
                "type": "function",
                "name": node.name,
                "file": self.filename,
                "module_path": self.module_path,  # ★ NEW
                "code": func_code,
                "meta": {},  # fill after visit
            }
        )

        self.func_index[func_id] = idx
        self.func_meta[func_id] = meta

        prev_func = self.current_func_id
        self.current_func_id = func_id
        try:
            self.generic_visit(node)
        finally:
            self.current_func_id = prev_func

        # finalize meta
        meta["attributes_used"] = sorted(meta["attributes_used"])
        meta["names_used"] = sorted(meta["names_used"])

        # imports used: intersect names/attributes with aliases
        used: Dict[str, str] = {}
        base_candidates = set(a.split(".", 1)[0] for a in meta["attributes_used"]) | set(meta["names_used"])
        for alias, full in self.import_alias.items():
            if alias in base_candidates:
                used[alias] = full
        meta["imports_used"] = used
        meta["metrics"]["n_calls"] = len(meta["calls_detailed"])

        code_chunks[idx]["meta"] = meta

    # -------- lambdas --------
    def visit_Lambda(self, node: ast.Lambda):
        """
        Visit a lambda expression.

        - Emits a lambda chunk with synthetic ID.
        - Captures arguments, body, enclosing function, and comments.
        - Links lambda ID into parent function metadata.
        """
        name = f"lambda@L{getattr(node, 'lineno', 0)}c{getattr(node, 'col_offset', 0)}"
        lam_id = f"{self.filename}::lambda::L{getattr(node, 'lineno', 0)}c{getattr(node, 'col_offset', 0)}"
        meta = {
            "args": _param_list(node.args),
            "body": _unparse(node.body),
            "enclosing": self.current_func_id,
            "comments": _comments_in_span(
                self.comments_map,
                getattr(node, "lineno", 0),
                getattr(node, "end_lineno", getattr(node, "lineno", 0)),
            ),
        }
        code_chunks.append(
            {
                "id": lam_id,
                "type": "lambda",
                "name": name,
                "file": self.filename,
                "module_path": self.module_path,  # ★ NEW
                "code": _get_source(self.source, node),
                "meta": meta,
            }
        )
        if self.current_func_id and self.current_func_id in self.func_meta:
            self.func_meta[self.current_func_id]["lambda_ids"].append(lam_id)
        self.generic_visit(node)

    # -------- returns / yields / exceptions --------
    def visit_Return(self, node: ast.Return):
        """
        Record return expressions inside current function and increment metrics.
        """

        if self.current_func_id and self.current_func_id in self.func_meta:
            self.func_meta[self.current_func_id]["return_values"].append(_unparse(node.value))
            self.func_meta[self.current_func_id]["metrics"]["n_returns"] += 1
        self.generic_visit(node)

    def visit_Yield(self, node: ast.Yield):
        """
        Record yield expressions and mark current function as a generator.
        """
        if self.current_func_id and self.current_func_id in self.func_meta:
            self.func_meta[self.current_func_id]["is_generator"] = True
            self.func_meta[self.current_func_id]["yield_values"].append(_unparse(node.value))
            self.func_meta[self.current_func_id]["metrics"]["n_yields"] += 1
        self.generic_visit(node)

    def visit_YieldFrom(self, node: ast.YieldFrom):
        """
        Record yield-from expressions and mark current function as a generator.
        """
        if self.current_func_id and self.current_func_id in self.func_meta:
            self.func_meta[self.current_func_id]["is_generator"] = True
            val = f"from { _unparse(node.value) }"
            self.func_meta[self.current_func_id]["yield_values"].append(val)
            self.func_meta[self.current_func_id]["metrics"]["n_yields"] += 1
        self.generic_visit(node)

    def visit_Raise(self, node: ast.Raise):
        """
        Record raise statements inside current function.
        """
        if self.current_func_id and self.current_func_id in self.func_meta:
            self.func_meta[self.current_func_id]["raises"].append(_unparse(node.exc))
        self.generic_visit(node)

    def visit_Try(self, node: ast.Try):
        """
        Record try/except blocks, exceptions handled, and branch metric.
        """
        if self.current_func_id and self.current_func_id in self.func_meta:
            self.func_meta[self.current_func_id]["metrics"]["n_branches"] += 1
            handled = []
            for h in node.handlers:
                handled.append(_unparse(h.type) or "Exception")
            self.func_meta[self.current_func_id]["exceptions_handled"].extend(handled)
        self.generic_visit(node)

    # -------- control flow metrics --------
    def visit_If(self, node: ast.If):
        """
        Increment branch count metric for if-statements.
        """
        if self.current_func_id and self.current_func_id in self.func_meta:
            self.func_meta[self.current_func_id]["metrics"]["n_branches"] += 1
        self.generic_visit(node)

    def visit_For(self, node: ast.For):
        """
        Increment branch count metric for for-loops.
        """
        if self.current_func_id and self.current_func_id in self.func_meta:
            self.func_meta[self.current_func_id]["metrics"]["n_branches"] += 1
        self.generic_visit(node)

    def visit_While(self, node: ast.While):
        """
        Increment branch count metric for while-loops.
        """
        if self.current_func_id and self.current_func_id in self.func_meta:
            self.func_meta[self.current_func_id]["metrics"]["n_branches"] += 1
        self.generic_visit(node)

    def visit_With(self, node: ast.With):
        """
        Increment branch count metric for with-statements.
        """
        if self.current_func_id and self.current_func_id in self.func_meta:
            self.func_meta[self.current_func_id]["metrics"]["n_branches"] += 1
        self.generic_visit(node)

    # -------- variable & attribute tracking --------
    def _record_local(
        self,
        name: str,
        annotation: Optional[str],
        value: Optional[ast.AST],
        type_comment: Optional[str],
        lineno: int,
    ):
        """
        Record assignment to a local variable in current function.
        """
        if self.current_func_id and self.current_func_id in self.func_meta:
            self.func_meta[self.current_func_id]["local_vars"].append(
                {
                    "name": name,
                    "annotation": annotation,
                    "inferred_type": _infer_type(value),
                    "value_preview": _value_preview(value),
                    "type_comment": type_comment,
                    "lineno": lineno,
                }
            )

    def _record_instance_attr(
        self,
        name: str,
        annotation: Optional[str],
        value: Optional[ast.AST],
        lineno: int,
        source: Optional[str] = None,
    ):
        """
        Record assignment to self.<attr> inside a method (instance attribute).
        """
        if self.current_func_id and self.current_func_id in self.func_meta:
            self.func_meta[self.current_func_id]["instance_attributes"].append(
                {
                    "name": name,
                    "source": source,
                    "annotation": annotation,
                    "inferred_type": _infer_type(value),
                    "value_preview": _value_preview(value),
                    "lineno": lineno,
                }
            )

    def visit_Assign(self, node: ast.Assign):
        """
        Visit assignment statements.
        - Distinguishes between instance attributes, locals, and tuple/list targets.
        """
        for tgt in node.targets:
            if isinstance(tgt, ast.Attribute) and isinstance(tgt.value, ast.Name) and tgt.value.id == "self":
                src = None
                if isinstance(node.value, ast.Name):
                    src = f"param: {node.value.id}"
                self._record_instance_attr(
                    tgt.attr, None, node.value, getattr(node, "lineno", 0), source=src
                )
            elif isinstance(tgt, ast.Name):
                self._record_local(
                    tgt.id, None, node.value, getattr(node, "type_comment", None), getattr(node, "lineno", 0)
                )
            elif isinstance(tgt, (ast.Tuple, ast.List)):
                for elt in tgt.elts:
                    if isinstance(elt, ast.Name):
                        self._record_local(
                            elt.id, None, None, getattr(node, "type_comment", None), getattr(node, "lineno", 0)
                        )
        self.generic_visit(node)

    def visit_AnnAssign(self, node: ast.AnnAssign):
        """
        Visit annotated assignment statements.
        - Records instance attributes or locals with type annotations.
        """
        ann = _unparse(node.annotation)
        if isinstance(node.target, ast.Attribute) and isinstance(node.target.value, ast.Name) and node.target.value.id == "self":
            self._record_instance_attr(node.target.attr, ann, node.value, getattr(node, "lineno", 0))
        elif isinstance(node.target, ast.Name):
            self._record_local(
                node.target.id, ann, node.value, getattr(node, "type_comment", None), getattr(node, "lineno", 0)
            )
        self.generic_visit(node)

    def visit_AugAssign(self, node: ast.AugAssign):
        """
        Visit augmented assignments (+=, -=, etc.).
        - Records updates to locals or instance attributes.
        """
        tgt = node.target
        if isinstance(tgt, ast.Attribute) and isinstance(tgt.value, ast.Name) and tgt.value.id == "self":
            self._record_instance_attr(tgt.attr, None, None, getattr(node, "lineno", 0))
        elif isinstance(tgt, ast.Name):
            self._record_local(tgt.id, None, None, None, getattr(node, "lineno", 0))
        self.generic_visit(node)

    # -------- names / attributes --------
    def visit_Attribute(self, node: ast.Attribute):
        """
        Record attribute usage inside current function.
        """
        if self.current_func_id and self.current_func_id in self.func_meta:
            dotted = _dotted_attr(node)
            if dotted:
                self.func_meta[self.current_func_id]["attributes_used"].add(dotted)
        self.generic_visit(node)

    def visit_Name(self, node: ast.Name):
        """
        Record variable name usage inside current function.
        """
        if self.current_func_id and self.current_func_id in self.func_meta:
            self.func_meta[self.current_func_id]["names_used"].add(node.id)
        self.generic_visit(node)

    # -------- calls --------
    def visit_Call(self, node: ast.Call):
        """
        Visit function/method calls.

        - Records call site details (callee, args, kwargs, lineno).
        - Adds call relation entry to global `call_relations`.
        - Updates function metadata (`calls_detailed`, metrics).
        """
        if self.current_func_id and self.current_func_id in self.func_meta:
            callee_full = _dotted_attr(node.func) or None
            if isinstance(node.func, ast.Name):
                callee_name = node.func.id
            elif isinstance(node.func, ast.Attribute):
                callee_name = node.func.attr
            elif callee_full:
                callee_name = callee_full.split(".")[-1]
            else:
                callee_name = None

            has_starargs = any(isinstance(a, ast.Starred) for a in node.args)
            has_kwargs = any(kw.arg is None for kw in node.keywords)

            self.func_meta[self.current_func_id]["calls_detailed"].append(
                {
                    "callee_fullname": callee_full,
                    "callee_name": callee_name,
                    "args": [_unparse(a) for a in node.args],
                    "keywords": {(kw.arg if kw.arg is not None else "**"): _unparse(kw.value) for kw in node.keywords},
                    "has_starargs": has_starargs,
                    "has_kwargs": has_kwargs,
                    "lineno": getattr(node, "lineno", None),
                }
            )

            if callee_name:
                call_relations.append(
                    {
                        "caller_id": self.current_func_id,
                        "callee_name": callee_name,
                        "callee_fullname": callee_full,
                        "lineno": getattr(node, "lineno", None),
                    }
                )
        self.generic_visit(node)

'convert_full_file'